# `parse_word` + `apply_changes` — pattern extract / modify / rebuild

Module testé : [`src/docpipeline/parsing/word/parse_word.py`](../src/docpipeline/parsing/word/parse_word.py).

## Le pattern transversal du projet

Pour traduction, conversion, redaction — partout où on doit modifier du texte SANS perdre les styles :

```
extract  : parse_word(docx)             → spans avec styles + span_id stable
modify   : on change SEULEMENT le text de chaque span (translate, redact, …)
rebuild  : apply_changes(docx_in, {span_id: new_text}, docx_out)
           → reconstruit le .docx avec nouveaux textes mais styles intacts
```

Le `span_id` (format `w_<para_idx>_<run_idx>`) est la **clé stable** qui fait le pont entre extract et rebuild — robuste aux textes dupliqués.

## Démo : cas d'usage **traduction FR → EN**

On parse `tests/fixtures/contrat_assurance.docx`, on traduit chaque span via un mini-dictionnaire FR→EN (sans LLM pour la démo — en prod ce serait un appel LLM, autorisé par `CLAUDE.md` dans la couche translation), et on reconstruit le `.docx` traduit.

Le fichier traduit est sauvegardé dans `notebooks/_outputs/word_translation/contrat_assurance_EN_js.docx` pour inspection visuelle dans Word.

In [1]:
# À exécuter une seule fois : installer le package en mode editable
import sys
!{sys.executable} -m pip install -q -e ..

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\DELL\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown
from docpipeline.parsing.word.parse_word import parse_word, apply_changes

DOCX_FR = Path('../tests/fixtures/contrat_assurance.docx')
DOCX_EN = Path('_outputs/word_translation/contrat_assurance_EN_js.docx')
DOCX_EN.parent.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_colwidth', 120)

# 1) EXTRACT --------------------------------------------------------------
result = parse_word(DOCX_FR)

display(Markdown('### 1. `doc_summary` — synthèse au niveau document'))
summary_df = pd.DataFrame(
    [{'key': k, 'value': str(v)} for k, v in result['doc_summary'].items()]
)
display(summary_df)

display(Markdown(f"### 2. `span_df` — {len(result['span_df'])} spans extraits avec leurs styles"))
display(result['span_df'][['span_id', 'text', 'font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color']])

display(Markdown(f"### 3. `paragraph_df` — {len(result['paragraph_df'])} paragraphes"))
display(result['paragraph_df'][['paragraph_index', 'text', 'style_name', 'heading_level', 'alignment', 'is_list_item']])

display(Markdown(f"### 4. `table_df` — {len(result['table_df'])} table(s) native(s)"))
display(result['table_df'])

### 1. `doc_summary` — synthèse au niveau document

,key,value
0,doc_hash,0134552717eb2b56e4ae7b799c060f7e70ed838968e293f3c9ea6c5faa9eb5c2
1,file_size_bytes,37367
2,n_paragraphs,10
3,n_spans,11
4,n_images,0
5,n_tables,1
6,n_sections,1
7,n_headings,6
8,n_list_items,0
9,total_char_count,689


### 2. `span_df` — 11 spans extraits avec leurs styles

,span_id,text,font_name,font_size_pt,bold,italic,underline,color
0,w_0_0,Contrat d'assurance — Conditions Générales,,None,False,False,None,NaN
1,w_1_0,1. Objet du contrat,,None,False,False,None,NaN
2,w_2_0,Le présent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activité professionnelle de l'a...,,None,False,False,None,NaN
3,w_3_0,2. Garanties,,None,False,False,None,NaN
4,w_4_0,2.1 Garantie principale,,None,False,False,None,NaN
5,w_5_0,La franchise applicable est fixée à 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par année...,,None,False,False,None,NaN
6,w_6_0,IMPORTANT :,,None,True,False,None,#FF0000
7,w_6_1,Toute déclaration de sinistre doit intervenir dans un délai de 5 jours ouvrés.,,None,False,False,None,NaN
8,w_7_0,3. Tableau des garanties,,None,False,False,None,NaN
9,w_8_0,4. Exclusions,,None,False,False,None,NaN


### 3. `paragraph_df` — 10 paragraphes

,paragraph_index,text,style_name,heading_level,alignment,is_list_item
0,0,Contrat d'assurance — Conditions Générales,Title,0.0,None,False
1,1,1. Objet du contrat,Heading 1,1.0,None,False
2,2,Le présent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activité professionnelle de l'a...,Normal,NaN,None,False
3,3,2. Garanties,Heading 1,1.0,None,False
4,4,2.1 Garantie principale,Heading 2,2.0,None,False
5,5,La franchise applicable est fixée à 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par année...,Normal,NaN,None,False
6,6,IMPORTANT : Toute déclaration de sinistre doit intervenir dans un délai de 5 jours ouvrés.,Normal,NaN,None,False
7,7,3. Tableau des garanties,Heading 1,1.0,None,False
8,8,4. Exclusions,Heading 1,1.0,None,False
9,9,"Sont exclus du présent contrat les sinistres résultant de : faute intentionnelle, guerre, actes terroristes, catastr...",Normal,NaN,None,False


### 4. `table_df` — 1 table(s) native(s)

,doc_hash,table_index,n_rows,n_cols,style_name,alignment,autofit,n_cells_with_text
0,0134552717eb2b56e4ae7b799c060f7e70ed838968e293f3c9ea6c5faa9eb5c2,0,5,3,Table Grid,None,True,15


In [3]:
# 2) MODIFY : traduction FR -> EN via dictionnaire (zero LLM pour la demo)
FR_TO_EN = {
    "Contrat d'assurance \u2014 Conditions G\u00e9n\u00e9rales": "Insurance Contract \u2014 General Conditions",
    "1. Objet du contrat": "1. Purpose of the contract",
    "Le pr\u00e9sent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activit\u00e9 professionnelle de l'assur\u00e9. La garantie BI (Business Interruption) est applicable selon l'article 5.": "This contract (IA) covers individual accidents occurring as part of the insured's professional activity. The BI (Business Interruption) coverage applies under article 5.",
    "2. Garanties": "2. Coverage",
    "2.1 Garantie principale": "2.1 Primary coverage",
    "La franchise applicable est fix\u00e9e \u00e0 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par ann\u00e9e civile.": "The applicable deductible is set at 300 euros per claim. The compensation limit is 50,000 euros per calendar year.",
    "IMPORTANT : ": "IMPORTANT: ",
    "Toute d\u00e9claration de sinistre doit intervenir dans un d\u00e9lai de 5 jours ouvr\u00e9s.": "Any claim must be reported within 5 business days.",
    "3. Tableau des garanties": "3. Coverage table",
    "4. Exclusions": "4. Exclusions",
    "Sont exclus du pr\u00e9sent contrat les sinistres r\u00e9sultant de : faute intentionnelle, guerre, actes terroristes, catastrophes naturelles non d\u00e9clar\u00e9es.": "The following claims are excluded from this contract: intentional fault, war, terrorism, undeclared natural disasters.",
}

changes = {row['span_id']: FR_TO_EN[row['text']]
           for _, row in result['span_df'].iterrows()
           if row['text'] in FR_TO_EN}

display(Markdown(f"### 5. Traductions appliquées : {len(changes)} / {len(result['span_df'])} spans"))
spans_idx = result['span_df'].set_index('span_id')
translations_df = pd.DataFrame([
    {'span_id': sid, 'FR (original)': spans_idx.loc[sid, 'text'], 'EN (traduit)': tr}
    for sid, tr in changes.items()
])
display(translations_df)

# 3) REBUILD
apply_changes(DOCX_FR, changes, DOCX_EN)
display(Markdown(f"**Fichier traduit écrit** : `{DOCX_EN}` ({DOCX_EN.stat().st_size} bytes)"))

### 5. Traductions appliquées : 11 / 11 spans

,span_id,FR (original),EN (traduit)
0,w_0_0,Contrat d'assurance — Conditions Générales,Insurance Contract — General Conditions
1,w_1_0,1. Objet du contrat,1. Purpose of the contract
2,w_2_0,Le présent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activité professionnelle de l'a...,This contract (IA) covers individual accidents occurring as part of the insured's professional activity. The BI (Bus...
3,w_3_0,2. Garanties,2. Coverage
4,w_4_0,2.1 Garantie principale,2.1 Primary coverage
5,w_5_0,La franchise applicable est fixée à 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par année...,"The applicable deductible is set at 300 euros per claim. The compensation limit is 50,000 euros per calendar year."
6,w_6_0,IMPORTANT :,IMPORTANT:
7,w_6_1,Toute déclaration de sinistre doit intervenir dans un délai de 5 jours ouvrés.,Any claim must be reported within 5 business days.
8,w_7_0,3. Tableau des garanties,3. Coverage table
9,w_8_0,4. Exclusions,4. Exclusions


**Fichier traduit écrit** : `_outputs\word_translation\contrat_assurance_EN_js.docx` (35993 bytes)

In [4]:
# 4) VERIF : on re-parse le .docx traduit et on compare avec l'original
result_en = parse_word(DOCX_EN)
before = result['span_df']
after  = result_en['span_df']

display(Markdown('### 6. Comparaison FR / EN — tous les spans côte à côte'))
compare_df = pd.DataFrame({
    'span_id':       before['span_id'].values,
    'FR (original)': before['text'].values,
    'EN (traduit)':  after['text'].values,
    'bold':          before['bold'].values,
    'italic':        before['italic'].values,
    'font_name':     before['font_name'].values,
})
display(compare_df)

display(Markdown('### 7. Styles préservés après traduction ?'))
checks = []
for col in ['font_name', 'font_size_pt', 'bold', 'italic', 'underline', 'color', 'highlight']:
    same = (before[col].fillna('').astype(str) == after[col].fillna('').astype(str)).all()
    checks.append({'attribute': col, 'preserved': 'OK' if same else 'DIFF'})
display(pd.DataFrame(checks))

### 6. Comparaison FR / EN — tous les spans côte à côte

,span_id,FR (original),EN (traduit),bold,italic,font_name
0,w_0_0,Contrat d'assurance — Conditions Générales,Insurance Contract — General Conditions,False,False,
1,w_1_0,1. Objet du contrat,1. Purpose of the contract,False,False,
2,w_2_0,Le présent contrat (IA) couvre les accidents individuels survenus dans le cadre de l'activité professionnelle de l'a...,This contract (IA) covers individual accidents occurring as part of the insured's professional activity. The BI (Bus...,False,False,
3,w_3_0,2. Garanties,2. Coverage,False,False,
4,w_4_0,2.1 Garantie principale,2.1 Primary coverage,False,False,
5,w_5_0,La franchise applicable est fixée à 300 euros par sinistre. Le plafond d'indemnisation est de 50 000 euros par année...,"The applicable deductible is set at 300 euros per claim. The compensation limit is 50,000 euros per calendar year.",False,False,
6,w_6_0,IMPORTANT :,IMPORTANT:,True,False,
7,w_6_1,Toute déclaration de sinistre doit intervenir dans un délai de 5 jours ouvrés.,Any claim must be reported within 5 business days.,False,False,
8,w_7_0,3. Tableau des garanties,3. Coverage table,False,False,
9,w_8_0,4. Exclusions,4. Exclusions,False,False,


### 7. Styles préservés après traduction ?

,attribute,preserved
0,font_name,OK
1,font_size_pt,OK
2,bold,OK
3,italic,OK
4,underline,OK
5,color,OK
6,highlight,OK
